# Return vs Generator (yield)

This notebook shows the difference between `return` and `yield` — the concept behind streaming.

## 1. Normal function with `return`

A normal function does ALL the work, then gives you EVERYTHING at once.

In [1]:
import time

def make_pizza_return():
    """Makes a pizza — you get nothing until ALL steps are done."""
    steps = []
    
    time.sleep(1)  # Making dough...
    steps.append("1. Dough ready")
    
    time.sleep(1)  # Adding sauce...
    steps.append("2. Sauce added")
    
    time.sleep(1)  # Adding cheese...
    steps.append("3. Cheese added")
    
    time.sleep(1)  # Baking...
    steps.append("4. Pizza baked!")
    
    return steps  # You waited 4 seconds and get everything at once

In [2]:
# Watch: nothing happens for 4 seconds, then all 4 steps print at once
print("Ordering pizza...")
start = time.time()

result = make_pizza_return()  # ← Stuck here for 4 seconds

print(f"Got result after {time.time() - start:.1f}s:")
for step in result:
    print(f"  {step}")

Ordering pizza...
Got result after 4.0s:
  1. Dough ready
  2. Sauce added
  3. Cheese added
  4. Pizza baked!


## 2. Generator function with `yield`

A generator gives you each piece AS SOON AS it's ready. You don't wait for everything.

In [3]:
def make_pizza_generator():
    """Makes a pizza — you get each step as soon as it's done."""
    
    time.sleep(1)
    yield "1. Dough ready"      # ← Give this to caller NOW, then continue
    
    time.sleep(1)
    yield "2. Sauce added"      # ← Give this NOW
    
    time.sleep(1)
    yield "3. Cheese added"     # ← Give this NOW
    
    time.sleep(1)
    yield "4. Pizza baked!"     # ← Give this NOW

In [4]:
# Watch: each step prints every 1 second (not all at once after 4s)
print("Ordering pizza...")
start = time.time()

for step in make_pizza_generator():  # ← Gets each yield one at a time
    print(f"  [{time.time() - start:.1f}s] {step}")

Ordering pizza...
  [1.0s] 1. Dough ready
  [2.0s] 2. Sauce added
  [3.0s] 3. Cheese added
  [4.0s] 4. Pizza baked!


## 3. See the type difference

In [5]:
# return gives you the actual data
result = make_pizza_return()
print(f"return gives: {type(result)}")
print(f"  value: {result}")
print()

# yield gives you a generator OBJECT (a lazy iterator)
gen = make_pizza_generator()
print(f"yield gives: {type(gen)}")
print(f"  value: {gen}")  # Not the data! Just a 'promise' to give data
print()
print("You have to LOOP over it or call next() to get values:")
print(f"  next(gen) = {next(gen)}")
print(f"  next(gen) = {next(gen)}")

return gives: <class 'list'>
  value: ['1. Dough ready', '2. Sauce added', '3. Cheese added', '4. Pizza baked!']

yield gives: <class 'generator'>
  value: <generator object make_pizza_generator at 0x10aff6d40>

You have to LOOP over it or call next() to get values:
  next(gen) = 1. Dough ready
  next(gen) = 2. Sauce added


## 4. The key insight: `yield` pauses the function

`yield` does something `return` can't — it **pauses** the function and **resumes** it later.

In [6]:
def count_to_3():
    print("  Starting...")
    yield 1
    print("  Resumed after yielding 1")
    yield 2
    print("  Resumed after yielding 2")
    yield 3
    print("  Done!")

print("Creating generator:")
gen = count_to_3()  # Nothing runs yet!
print(f"  (nothing happened yet)\n")

print("Calling next(gen) first time:")
val = next(gen)
print(f"  Got: {val}\n")

print("Calling next(gen) second time:")
val = next(gen)
print(f"  Got: {val}\n")

print("Calling next(gen) third time:")
val = next(gen)
print(f"  Got: {val}\n")

Creating generator:
  (nothing happened yet)

Calling next(gen) first time:
  Starting...
  Got: 1

Calling next(gen) second time:
  Resumed after yielding 1
  Got: 2

Calling next(gen) third time:
  Resumed after yielding 2
  Got: 3



## 5. How this connects to LLM streaming

The LLM streaming code is the same pattern — just with `async`:

In [7]:
# Fake LLM that generates tokens one at a time
def fake_llm_stream(question):
    """Pretend to be an LLM generating tokens."""
    answer = f"The answer to '{question}' is that Python is great."
    words = answer.split(" ")
    
    for i, word in enumerate(words):
        time.sleep(0.3)  # Simulate LLM thinking time
        token = f" {word}" if i > 0 else word
        yield token  # Hand one token to the caller


# This is what your streaming code does:
print("LLM response: ", end="")
for token in fake_llm_stream("What is Python?"):
    print(token, end="", flush=True)  # Print each token as it arrives
print("\n\n(Each word appeared one at a time!)")

LLM response: The answer to 'What is Python?' is that Python is great.

(Each word appeared one at a time!)


## 6. return vs yield — Summary

| | `return` | `yield` |
|---|---|---|
| Gives you | Everything at once | One piece at a time |
| Function | Runs to completion, then returns | Pauses at each yield, resumes on next() |
| Memory | Builds full list in memory | Only 1 item in memory at a time |
| Use case | Small data, need all at once | Streaming, large data, lazy evaluation |
| Analogy | Waiter brings full meal | Waiter brings each dish as it's cooked |

In [8]:
# One more example to make it click:

# return: like downloading a full movie, then watching
def download_movie():
    frames = []
    for i in range(100):
        frames.append(f"frame_{i}")
    return frames  # All 100 frames at once

# yield: like streaming a movie (watch while downloading)
def stream_movie():
    for i in range(100):
        yield f"frame_{i}"  # One frame at a time

# The result is the same — but WHEN you get it differs
print("Download: get all frames, then display")
all_frames = download_movie()
print(f"  Got {len(all_frames)} frames at once: {all_frames[:3]}...\n")

print("Stream: get one frame, display it, get next")
for i, frame in enumerate(stream_movie()):
    if i < 3:
        print(f"  Playing: {frame}")
    elif i == 3:
        print(f"  ... (97 more frames coming one by one)")
        break

Download: get all frames, then display
  Got 100 frames at once: ['frame_0', 'frame_1', 'frame_2']...

Stream: get one frame, display it, get next
  Playing: frame_0
  Playing: frame_1
  Playing: frame_2
  ... (97 more frames coming one by one)
